# FWK-BVCS-SVMR seismic response prediction

Predict MIDR and MFA using the 14 IM-Corr inputs and structural parameters.
Run all cells from this directory. Data are in data/ and results are saved in results/.


## 1. Imports, selected inputs and runtime configuration

The selected IMs are SI, EPV, VSI, SAavg, PGV, PSV, Sa, Vrms, PGA, ASI, EPA,
PSA, Ia and Arms, in the original order. Structural and brace inputs are retained.
`M` corresponds to `S` in the manuscript. Fixed parameters are retained from the
original notebook; `RUN_CV` optionally recomputes the main parameter search.

BLAS and MI concurrency below were measured on an AMD Ryzen 9 7945HX.
Rebenchmark on another computer; a larger thread count is not always faster.


In [ ]:
import gc
import json
import time
from contextlib import contextmanager
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from joblib import parallel_config
from matplotlib import rcParams
from matplotlib.ticker import FuncFormatter, MultipleLocator
from sklearn.cluster import kmeans_plusplus
from sklearn.feature_selection import mutual_info_regression, r_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

get_ipython().run_line_magic("matplotlib", "inline")

In [ ]:
DATA_PATH = Path("data/Data_clean_english.xlsx")
TARGET_COLS = ["MIDR", "MFA"]
SPLIT_SEED = 42
ARCH_COLS = [
    "stories",
    "first_story_height_m",
    "typical_story_height_m",
    "span_length_m",
    "column_section_dim1_m",
    "fundamental_period_rcf_s",
    "beam_section_dim2_m",
]
PDYBRB_COLS = [
    "apdybrb_fy2_mean",
    "apdybrb_fy2_std",
    "apdybrb_fy2_max",
    "apdybrb_fy2_min",
    "apdybrb_fy2_sum",
    "apdybrb_fy2_var",
    "first_stage_yield_ratio",
    "brace_arrangement",
    "apdybrb_activation_disp_mean",
    "apdybrb_activation_disp_std",
    "apdybrb_activation_disp_max",
    "apdybrb_activation_disp_min",
    "apdybrb_activation_disp_sum",
    "apdybrb_activation_disp_var",
]
CAT_COLS = ["stories", "brace_arrangement", "first_stage_yield_ratio"]
CATEGORY_MAPS = {
    "stories": {5: 0, 7: 1, 9: 2, 11: 3},
    "brace_arrangement": {1: 0, 2: 1, 3: 2},
    "first_stage_yield_ratio": {0.4: 0, 0.5: 1},
}
IM_SCHEMES = {
    "IM-Corr": [
        "SI (cm)",
        "EPV (cm/s)",
        "VSI (cm)",
        "SAavg (g)",
        "PGV (cm/s)",
        "PSV (cm/s)",
        "Sa (g)",
        "Vrms (cm/s)",
        "PGA (g)",
        "ASI (cm/s)",
        "EPA (g)",
        "PSA (g)",
        "Ia (m/s)",
        "Arms (g)",
    ]
}
FWK_MODEL_NAME = "FWK-BVCS-SVMR"
RESULTS_ROOT = Path("results")
FWK_SCHEMES_TO_RUN = ["IM-Corr"]
TRAIN_SEED = 42
M = 1500
RUN_BASIS_CONVERGENCE = True
BASIS_COUNTS = [100, 250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2500, 3000]
BASIS_CV_FOLDS = 5
W_SAMPLE_SIZE = 3000
PREDICT_BATCH_SIZE = 2048
RUN_CV = False
GLOBAL_PARAMS = {"sig2": 100.0, "gamma": 100.0}
CV_FOLDS = 5
SIG2_GRID = [20.0, 50.0, 100.0, 200.0, 400.0, 800.0]
GAMMA_GRID = [1.0, 100.0, 10000.0]
ENABLE_LOG_FEATURES = True
MODEL_NAME = FWK_MODEL_NAME
data_root = RESULTS_ROOT / MODEL_NAME
fig_dir = data_root / "figures"
TARGET_LABELS = ["MIDR", "MFA"]
AXIS_CONFIG = {
    "MIDR": {"limit": 0.09, "step": 0.02, "integer": False, "diag_end": 1.0},
    "MFA": {"limit": 20, "step": 5, "integer": True, "diag_end": 25.0},
}
rcParams["font.family"] = "serif"
rcParams["font.serif"] = ["Times New Roman"]
rcParams["axes.unicode_minus"] = False
rcParams["mathtext.fontset"] = "stix"
FWK_RUN_DIR = RESULTS_ROOT / FWK_MODEL_NAME
RUN_PREDICTION_PLOTS = True
BLAS_THREADS = 16
MI_N_JOBS = 16
OMP_THREADS = 1

In [ ]:
@contextmanager
def compute_runtime():
    """Apply the measured BLAS/MI policy without changing global thread settings."""
    with (
        threadpool_limits(limits=BLAS_THREADS, user_api="blas"),
        threadpool_limits(limits=OMP_THREADS, user_api="openmp"),
        parallel_config(backend="threading", n_jobs=MI_N_JOBS),
    ):
        yield

## 2. Data preparation and fixed split

Keep the original filter: exclude rows with MFA > 15 or MIDR > 0.2.
Hold out stories/PGA pairs (5, 0.5), (7, 0.4), (9, 0.3), (11, 0.2) as Test2.
Split the remaining rows 70/30 into Train and Test1 using seed 42.
Fit standardization using Train only. The original one-hot treatment of
`first_stage_yield_ratio` is retained alongside stories and arrangement.
`sample_id` is the zero-based row index after filtering, as in the source.


In [ ]:
data = pd.read_excel(DATA_PATH)

data = data[~((data["MFA"] > 15) | (data["MIDR"] > 0.2))]

data = data.reset_index(drop=True)


data.columns = data.columns.astype(str).str.strip()

if data.columns.duplicated().any():
    raise ValueError("The dataset contains duplicate column names.")

target_cols = data.columns[-2:].tolist() if TARGET_COLS is None else list(TARGET_COLS)

base_cols = ARCH_COLS + PDYBRB_COLS

for col in ["stories", "brace_arrangement"]:
    data[col] = pd.to_numeric(data[col], errors="raise")

ratio = pd.to_numeric(
    data["first_stage_yield_ratio"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False),
    errors="raise",
)

data["first_stage_yield_ratio"] = np.round(np.where(ratio > 1, ratio / 100, ratio), 6)

onehot_blocks, onehot_names = [], []

for col in CAT_COLS:
    mapping = CATEGORY_MAPS[col]
    codes = data[col].map(mapping)
    if codes.isna().any():
        raise ValueError(f"{col} contains missing values or unknown categories.")
    onehot_blocks.append(np.eye(len(mapping))[codes.to_numpy(dtype=np.int64)])
    onehot_names.extend(f"{col}={value}" for value in mapping)

x_onehot = np.column_stack(onehot_blocks)

all_inputs = list(
    dict.fromkeys(base_cols + [col for cols in IM_SCHEMES.values() for col in cols])
)

if len(target_cols) != 2 or len(set(target_cols)) != 2:
    raise ValueError("TARGET_COLS must contain two distinct EDP columns.")

if set(target_cols) & set(all_inputs):
    raise ValueError("Target columns overlap with input columns.")

missing = [col for col in all_inputs + target_cols if col not in data.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

continuous_cols = [col for col in all_inputs if col not in CAT_COLS]

check_cols = continuous_cols + target_cols

data[check_cols] = data[check_cols].apply(pd.to_numeric, errors="raise")

if not np.isfinite(data[check_cols].to_numpy(dtype=float)).all():
    raise ValueError("Inputs or targets contain NaN/Inf.")

test2_mask = np.zeros(len(data), dtype=bool)

for story, intensity in [(5, 0.5), (7, 0.4), (9, 0.3), (11, 0.2)]:
    test2_mask |= (data["stories"].to_numpy() == story) & np.isclose(
        data["PGA (g)"].to_numpy(), intensity, rtol=0, atol=1e-6
    )

test2_idx = np.flatnonzero(test2_mask)

remaining_idx = np.flatnonzero(~test2_mask)

if not len(test2_idx):
    raise ValueError("Test2 is empty; check stories and PGA.")

train_idx, test1_idx = train_test_split(
    remaining_idx, test_size=0.3, random_state=SPLIT_SEED
)

split_indices = {"Train": train_idx, "Test1": test1_idx, "Test2": test2_idx}

split_manifest = pd.DataFrame({"sample_id": np.arange(len(data)), "split": ""})

for split_name, ids in split_indices.items():
    split_manifest.loc[ids, "split"] = split_name

y_raw = data[target_cols].to_numpy(dtype=float)

scaler_y = StandardScaler().fit(y_raw[train_idx])

y_scaled = scaler_y.transform(y_raw)

story_raw = data["stories"].to_numpy(dtype=float)

fwk_data = {}

for scheme_name, im_cols in IM_SCHEMES.items():
    if len(im_cols) != len(set(im_cols)):
        raise ValueError(f"{scheme_name} contains duplicate IMs.")
    num_cols = [col for col in base_cols + im_cols if col not in CAT_COLS]
    x_num_raw = data[num_cols].to_numpy(dtype=float)
    if not np.isfinite(x_num_raw).all():
        raise ValueError(f"{scheme_name} inputs contain NaN/Inf.")
    scaler_x = StandardScaler().fit(x_num_raw[train_idx])
    x = np.column_stack([scaler_x.transform(x_num_raw), x_onehot])
    fwk_data[scheme_name] = {
        "im_cols": list(im_cols),
        "num_cols": num_cols,
        "feature_names": num_cols + onehot_names,
        "discrete_features": np.array(
            [False] * len(num_cols) + [True] * x_onehot.shape[1]
        ),
        "scaler_x": scaler_x,
        "scaler_y": scaler_y,
        "splits": {
            name: {
                "X": x[ids],
                "Y": y_scaled[ids],
                "y_raw": y_raw[ids],
                "x_num_raw": x_num_raw[ids],
                "x_onehot": x_onehot[ids],
                "story_raw": story_raw[ids],
                "sample_id": ids.copy(),
            }
            for name, ids in split_indices.items()
        },
    }

## 3. Weighted kernel and shared multi-output model

The kernel acts on a training-fitted dictionary containing identity coordinates,
nonlinear transforms and interactions. MI is normalized by the largest MI
across coordinates for each target, then combined with absolute Pearson
correlation. Coordinate weights are shared across outputs. One reduced linear
system estimates the two coefficient columns and the two biases.

`prototype_selection` supports `kmeans`, `kmeans++`, `farthest` and `random`.
The default `kmeans++` selects training samples without Lloyd iterations.


In [ ]:
def weighted_rbf_kernel(X1, X2, sig2, weights):
    if not np.isfinite(sig2) or sig2 <= 0:
        raise ValueError("sig2 must be finite and positive.")
    X1_w, X2_w = X1 * np.sqrt(weights), X2 * np.sqrt(weights)
    dists = (
        np.sum(X1_w**2, axis=1)[:, None]
        + np.sum(X2_w**2, axis=1)[None, :]
        - 2 * X1_w @ X2_w.T
    )
    return np.exp(-np.maximum(dists, 0.0) / (2 * sig2))


class FWKBVCSSVMR:
    """Multi-output reduced kernel regression with a training-fitted dictionary.

    The weighted RBF kernel acts on original and derived coordinates.
    Shared feature weights and basis vectors are fitted from training data.
    """

    def __init__(
        self,
        sig2=1.0,
        gamma=1.0,
        M=1500,
        seed=42,
        weight_method="combined",
        w_sample_size=3000,
        discrete_features=None,
        num_names=None,
        x_mean=None,
        x_std=None,
        base_feature_names=None,
        n_scheme_im=0,
    ):
        self.sig2, self.gamma, self.M = sig2, gamma, M
        self.seed = seed
        self.weight_method, self.w_sample_size = weight_method, w_sample_size
        self.discrete_features = discrete_features

        self.num_names = list(num_names) if num_names is not None else []
        self.x_mean = None if x_mean is None else np.asarray(x_mean, dtype=float)
        self.x_std = None if x_std is None else np.asarray(x_std, dtype=float)
        self.base_feature_names = list(base_feature_names or [])
        self.n_scheme_im = n_scheme_im

    def _dictionary_coords(self, X, story_raw, fitting):
        """FWK FWK construction."""
        n_num = len(self.num_names)
        X_raw = X[:, :n_num] * self.x_std + self.x_mean
        v = {name: X_raw[:, i] for i, name in enumerate(self.num_names)}
        stories = np.asarray(story_raw, dtype=float)
        outs, names, parents = [], [], []
        if fitting:
            self._dict_pos_ = {}

        def add(arr, name, src):
            outs.append(np.asarray(arr, dtype=float))
            names.append(name)
            parents.append(tuple(src))

        def family(vals, base_name, src):
            """Apply log, square root and square transforms; fit positivity once."""
            if fitting:
                self._dict_pos_[base_name] = bool(
                    ENABLE_LOG_FEATURES and np.all(vals > 0)
                )
            if self._dict_pos_[base_name]:
                add(np.log(np.maximum(vals, 0.0) + 1e-6), f"log_{base_name}", src)
                add(np.sqrt(np.maximum(vals, 0.0)), f"sqrt_{base_name}", src)
            add(vals**2, f"{base_name}_squared", src)

        inter = []
        if "PGA (g)" in v:
            inter = [
                (stories * v["PGA (g)"], "stories_x_PGA"),
                ((stories**2) * v["PGA (g)"], "stories_squared_x_PGA"),
                (stories * v["PGA (g)"] ** 2, "stories_x_PGA_squared"),
            ]
            for arr, nm in inter:
                add(arr, nm, ["stories", "PGA (g)"])
        add(stories**2, "stories_squared", ["stories"])
        add(stories**3, "stories_cubed", ["stories"])
        add(1.0 / stories, "1/stories", ["stories"])
        if "fundamental_period_rcf_s" in v:
            add(
                stories * v["fundamental_period_rcf_s"],
                "stories_x_T1",
                ["stories", "fundamental_period_rcf_s"],
            )

        for col in self.num_names:
            family(v[col], col, [col])

        for arr, nm in inter:
            family(arr, nm, ["stories", "PGA (g)"])

        if "PGA (g)" in v and "PGV (cm/s)" in v:
            add(
                np.where(
                    v["PGA (g)"] > 1e-6, v["PGV (cm/s)"] / (v["PGA (g)"] * 980.0), 0.0
                ),
                "pgv_pga_period_proxy",
                ["PGA (g)", "PGV (cm/s)"],
            )
        if "PGA (g)" in v and "fundamental_period_rcf_s" in v:
            add(
                v["PGA (g)"] * np.power(v["fundamental_period_rcf_s"] + 0.1, -0.5),
                "pga_period_proxy",
                ["PGA (g)", "fundamental_period_rcf_s"],
            )

        if "apdybrb_fy2_mean" in v and "PGA (g)" in v:
            add(
                v["apdybrb_fy2_mean"] * v["PGA (g)"],
                "apdybrb_fy2_x_PGA",
                ["apdybrb_fy2_mean", "PGA (g)"],
            )
            add(
                np.where(
                    v["PGA (g)"] > 1e-6, v["apdybrb_fy2_mean"] / v["PGA (g)"], 0.0
                ),
                "apdybrb_fy2/PGA",
                ["apdybrb_fy2_mean", "PGA (g)"],
            )

        if "apdybrb_activation_disp_mean" in v and "PGA (g)" in v:
            add(
                v["apdybrb_activation_disp_mean"] * v["PGA (g)"],
                "activation_disp_x_PGA",
                ["apdybrb_activation_disp_mean", "PGA (g)"],
            )

        D = np.column_stack(outs) if outs else np.empty((len(X), 0))
        if fitting:
            self.derived_names_ = names
            self.derived_parents_ = parents
            self.d_mean_ = D.mean(axis=0) if D.shape[1] else np.empty(0)
            d_std = D.std(axis=0) if D.shape[1] else np.empty(0)
            self.d_std_ = np.where(d_std < 1e-12, 1.0, d_std)
        elif names != list(getattr(self, "derived_names_", [])):
            raise ValueError("Prediction dictionary coordinates differ from training.")
        return (D - self.d_mean_) / self.d_std_ if D.shape[1] else D

    def _lift(self, X, story_raw, fitting=False):
        D = self._dictionary_coords(X, story_raw, fitting)
        if not D.shape[1]:
            return X
        n_base = len(self.num_names) - self.n_scheme_im

        return np.column_stack([X[:, :n_base], D, X[:, n_base:]])

    def variable_weight_report(self, weights=None):
        """Aggregate coordinate weights to their source variables."""
        weights = self.feature_weights if weights is None else weights
        n_base, n_derived = self._n_base_, len(self.derived_names_)
        agg = {}

        def var_of(col):
            return col.split("=")[0] if "=" in col else col

        for name, pos in zip(self.base_feature_names, self._orig_pos_):
            w = float(weights[pos])
            agg[var_of(name)] = agg.get(var_of(name), 0.0) + w
        for name, w, ps in zip(
            self.derived_names_,
            weights[n_base : n_base + n_derived],
            self.derived_parents_,
        ):
            for p in ps:
                agg[p] = agg.get(p, 0.0) + float(w) / len(ps)
        return sorted(agg.items(), key=lambda t: -t[1])

    def _compute_weights_optimized(self, X, Y):
        rng = np.random.RandomState(self.seed)
        if len(X) > self.w_sample_size:
            ids = rng.choice(len(X), self.w_sample_size, replace=False)
            X_w, Y_w = X[ids], Y[ids]
        else:
            X_w, Y_w = X, Y
        linear = np.zeros((X.shape[1], Y.shape[1]))
        mi = np.zeros_like(linear)
        variable = np.ptp(X_w, axis=0) > 0
        if variable.any():
            for j in range(Y.shape[1]):
                if self.weight_method in ["linear", "combined"]:
                    linear[variable, j] = np.abs(
                        r_regression(X_w[:, variable], Y_w[:, j])
                    )
                if self.weight_method in ["mi", "combined"]:
                    mi[variable, j] = mutual_info_regression(
                        X_w[:, variable],
                        Y_w[:, j],
                        discrete_features=self.discrete_features_[variable],
                        random_state=self.seed,
                    )
        linear = np.clip(np.nan_to_num(linear), 0, 1)
        if self.weight_method == "linear":
            scores = linear
        elif self.weight_method == "mi":
            scores = mi
        else:
            maxima = mi.max(axis=0, keepdims=True)
            normalized_mi = np.divide(
                mi, maxima, out=np.zeros_like(mi), where=maxima > 0
            )
            scores = (linear + normalized_mi) / 2
        weights = np.mean(scores, axis=1) ** 2 + 0.05
        return weights / weights.max()

    def _prepare_basis(self, X, Y, story_raw, weights=None, Xs=None):
        if X.ndim != 2 or Y.ndim != 2 or len(X) != len(Y):
            raise ValueError("X and Y must be 2D arrays with equal row counts.")
        if not np.isfinite(X).all() or not np.isfinite(Y).all():
            raise ValueError("X or Y contains NaN/Inf.")
        if not isinstance(self.M, (int, np.integer)) or not 1 <= self.M <= len(X):
            raise ValueError(
                f"M={self.M} must be an integer from 1 to {len(X)} (number of training samples)."
            )
        if self.w_sample_size < 4:
            raise ValueError("w_sample_size must be at least 4.")
        if self.weight_method not in ["linear", "mi", "combined"]:
            raise ValueError("weight_method must be linear, mi or combined.")
        self.n_features_in_ = X.shape[1]

        X_aug = self._lift(X, story_raw, fitting=True)
        n_derived = X_aug.shape[1] - X.shape[1]
        base_discrete = (
            np.zeros(X.shape[1], dtype=bool)
            if self.discrete_features is None
            else np.asarray(self.discrete_features, dtype=bool)
        )
        if base_discrete.shape != (X.shape[1],):
            raise ValueError(
                "discrete_features length does not match the input dimension."
            )
        n_base = len(self.num_names) - self.n_scheme_im
        self._n_base_ = n_base
        self.discrete_features_ = np.concatenate(
            [
                base_discrete[:n_base],
                np.zeros(n_derived, dtype=bool),
                base_discrete[n_base:],
            ]
        )
        self.n_features_aug_ = X_aug.shape[1]

        self._orig_pos_ = list(range(n_base)) + list(
            range(n_base + n_derived, self.n_features_aug_)
        )

        self.feature_weights = (
            self._compute_weights_optimized(X_aug, Y)
            if weights is None
            else np.asarray(weights, dtype=float)
        )
        if (
            self.feature_weights.shape != (X_aug.shape[1],)
            or not np.isfinite(self.feature_weights).all()
            or (self.feature_weights < 0).any()
            or not (self.feature_weights > 0).any()
        ):
            raise ValueError(
                "Feature weights must match the lifted dimension and contain valid nonnegative values."
            )
        if Xs is not None:
            self.Xs = np.asarray(Xs, dtype=float)
            if self.Xs.shape[1] != self.n_features_aug_:
                raise ValueError(
                    "Injected basis dimension differs from the lifted input dimension."
                )
        else:
            sqrt_w = np.sqrt(self.feature_weights)
            X_w = X_aug * sqrt_w
            _, ids = kmeans_plusplus(X_w, n_clusters=self.M, random_state=self.seed)
            self.Xs = X_aug[ids]
        return X_aug

    def _normal_equations(self, X, Y):
        K = weighted_rbf_kernel(self.Xs, self.Xs, self.sig2, self.feature_weights)
        K_e = weighted_rbf_kernel(self.Xs, X, self.sig2, self.feature_weights)
        K_i = np.vstack([K_e, np.ones((1, len(X)))])
        return K, K_i @ K_i.T, K_i @ Y

    def _solve(self, equations):
        if not np.isfinite(self.gamma) or self.gamma <= 0:
            raise ValueError("gamma must be finite and positive.")
        K, gram, rhs = equations
        A = gram.copy()
        m = len(self.Xs)
        A[:m, :m] += (K + K.T) / (2 * self.gamma)
        A.flat[:: A.shape[0] + 1] += 1e-8
        coefficients = np.linalg.solve(A, rhs)
        self.beta = coefficients[:m]
        self.b = coefficients[-1]

    def fit(self, X_train, Y_train, story_raw, weights=None, Xs=None):
        X = np.asarray(X_train, dtype=float)
        Y = np.asarray(Y_train, dtype=float)
        if Y.ndim == 1:
            Y = Y[:, None]
        X_aug = self._prepare_basis(X, Y, story_raw, weights=weights, Xs=Xs)
        self._solve(self._normal_equations(X_aug, Y))
        return self

    def predict(self, X_test, story_raw, batch_size=2048):
        X_test = np.asarray(X_test, dtype=float)
        if X_test.ndim != 2 or X_test.shape[1] != self.n_features_in_:
            raise ValueError("Prediction input dimension differs from training.")
        if batch_size < 1 or not np.isfinite(X_test).all():
            raise ValueError(
                "Prediction input is non-finite or batch_size is less than 1."
            )
        X_aug = self._lift(X_test, story_raw, fitting=False)
        predictions = np.empty((len(X_aug), self.beta.shape[1]))
        for start in range(0, len(X_aug), batch_size):
            stop = min(start + batch_size, len(X_aug))
            K_t = weighted_rbf_kernel(
                self.Xs, X_aug[start:stop], self.sig2, self.feature_weights
            )
            predictions[start:stop] = K_t.T @ self.beta + self.b
        return predictions


def new_fwk(package, **parameters):
    return FWKBVCSSVMR(
        M=M,
        seed=TRAIN_SEED,
        weight_method="combined",
        w_sample_size=W_SAMPLE_SIZE,
        discrete_features=package["discrete_features"],
        num_names=package["num_cols"],
        x_mean=package["scaler_x"].mean_,
        x_std=package["scaler_x"].scale_,
        base_feature_names=package["feature_names"],
        n_scheme_im=len(package["im_cols"]),
        **parameters,
    )


def make_shared_basis(package, X, Y_joint, story_raw):
    """Fit one shared set of weights and basis vectors for all outputs."""
    probe = new_fwk(package)
    probe._prepare_basis(X, Y_joint, story_raw)
    return probe


def _attach_condition_groups(package):
    """Add stories/PGA group labels for condition-based validation."""
    story_values = list(CATEGORY_MAPS["stories"])
    n_story = len(story_values)
    pga_pos = (
        package["num_cols"].index("PGA (g)")
        if "PGA (g)" in package["num_cols"]
        else None
    )
    for arrays in package["splits"].values():
        story_idx = np.argmax(arrays["x_onehot"][:, :n_story], axis=1)
        if pga_pos is not None:
            pga = np.round(arrays["x_num_raw"][:, pga_pos], 6)
            labels = [f"{story_values[s]}@{p}" for s, p in zip(story_idx, pga)]
        else:
            labels = [str(story_values[s]) for s in story_idx]
        arrays["group"] = pd.factorize(pd.Series(labels))[0]


def _grouped_folds(train):
    """Use GroupKFold when enough groups exist; otherwise use KFold."""
    groups = train.get("group")
    if groups is not None:
        n_groups = len(np.unique(groups))
        if n_groups >= CV_FOLDS:
            return list(
                GroupKFold(n_splits=CV_FOLDS).split(train["y_raw"], groups=groups)
            )
    return list(
        KFold(n_splits=CV_FOLDS, shuffle=True, random_state=TRAIN_SEED).split(
            train["y_raw"]
        )
    )


def _mixed_folds(train):
    """Combine shuffled KFold and condition-based GroupKFold splits."""
    kfold = list(
        KFold(n_splits=CV_FOLDS, shuffle=True, random_state=TRAIN_SEED).split(
            train["y_raw"]
        )
    )
    return [("kfold", f) for f in kfold] + [
        ("grouped", f) for f in _grouped_folds(train)
    ]


def _tune_mixed(package):
    """Select parameters using the mean of KFold and grouped mean R2.

    Input scaling, dictionary statistics, weights and bases are fitted within
    each fold. Target scaling follows the original outer-training scaler."""
    train = package["splits"]["Train"]
    t0_name, t1_name = target_cols[0], target_cols[1]
    rows = []
    for regime, (fit_ids, valid_ids) in _mixed_folds(train):
        sx = StandardScaler().fit(train["x_num_raw"][fit_ids])
        X_fit = np.column_stack(
            [sx.transform(train["x_num_raw"][fit_ids]), train["x_onehot"][fit_ids]]
        )
        X_valid = np.column_stack(
            [sx.transform(train["x_num_raw"][valid_ids]), train["x_onehot"][valid_ids]]
        )
        Y_fit, Y_valid = train["Y"][fit_ids], train["Y"][valid_ids]
        fold_package = dict(package)
        fold_package["scaler_x"] = sx

        probe = make_shared_basis(
            fold_package, X_fit, Y_fit, train["story_raw"][fit_ids]
        )
        model = new_fwk(fold_package)
        model._prepare_basis(
            X_fit,
            Y_fit,
            train["story_raw"][fit_ids],
            weights=probe.feature_weights,
            Xs=probe.Xs,
        )
        X_fit_aug = model._lift(X_fit, train["story_raw"][fit_ids])
        for sig2 in SIG2_GRID:
            model.sig2 = sig2
            equations = model._normal_equations(X_fit_aug, Y_fit)
            for gamma in GAMMA_GRID:
                model.gamma = gamma
                model._solve(equations)
                r2_each = r2_score(
                    Y_valid,
                    model.predict(
                        X_valid, train["story_raw"][valid_ids], PREDICT_BATCH_SIZE
                    ),
                    multioutput="raw_values",
                )
                rows.append(
                    {
                        "regime": regime,
                        "sig2": sig2,
                        "gamma": gamma,
                        f"R2_{t0_name}": float(r2_each[0]),
                        f"R2_{t1_name}": float(r2_each[1]),
                        "R2_mean": float(r2_each.mean()),
                    }
                )
        print(f"    {regime} fold completed ({len(rows)} records)")
    fold_scores = pd.DataFrame(rows)
    agg = fold_scores.groupby(["sig2", "gamma"])
    summary = agg.apply(
        lambda g: pd.Series(
            {
                "kfold_R2": g.loc[g["regime"] == "kfold", "R2_mean"].mean(),
                "grouped_R2": g.loc[g["regime"] == "grouped", "R2_mean"].mean(),
            }
        ),
        include_groups=False,
    ).reset_index()
    summary["mixed_R2"] = (summary["kfold_R2"] + summary["grouped_R2"]) / 2
    summary = summary.sort_values(
        ["mixed_R2", "sig2", "gamma"], ascending=[False, True, True]
    )
    best = summary.iloc[0]
    params = {"sig2": float(best["sig2"]), "gamma": float(best["gamma"])}
    return params, summary, fold_scores


def _boundary_warning(params, sig2_grid, gamma_grid, label):
    hits = []
    if params["sig2"] in [min(sig2_grid), max(sig2_grid)]:
        hits.append(f"sig2={params['sig2']}")
    if params["gamma"] in [min(gamma_grid), max(gamma_grid)]:
        hits.append(f"gamma={params['gamma']}")
    if hits:
        print(
            f"  Note: {label} selected parameters are on the grid boundary ({', '.join(hits)}); consider checking a wider grid."
        )

## 4. Training and prediction

Training time includes training-only parameter selection, weighted-kernel learning,
basis selection and final fitting. Prediction time covers predict for both outputs.
Data loading, outer preprocessing, metrics, export and plotting are excluded.


In [ ]:
with compute_runtime():
    if not FWK_SCHEMES_TO_RUN or any(
        (name not in fwk_data for name in FWK_SCHEMES_TO_RUN)
    ):
        raise ValueError("FWK_SCHEMES_TO_RUN is empty or contains unknown schemes.")
    if RUN_CV and (
        not SIG2_GRID
        or not GAMMA_GRID
        or any((not np.isfinite(v) or v <= 0 for v in SIG2_GRID + GAMMA_GRID))
    ):
        raise ValueError(
            "SIG2_GRID and GAMMA_GRID must contain finite positive values."
        )
    if not RUN_CV and (GLOBAL_PARAMS["sig2"] <= 0 or GLOBAL_PARAMS["gamma"] <= 0):
        raise ValueError("GLOBAL_PARAMS requires positive sig2 and gamma.")
    for _scheme in FWK_SCHEMES_TO_RUN:
        _attach_condition_groups(fwk_data[_scheme])
    FWK_RUN_DIR = RESULTS_ROOT / "FWK-BVCS-SVMR"
    FWK_RUN_DIR.mkdir(parents=True, exist_ok=True)
    split_manifest.to_csv(
        FWK_RUN_DIR / "sample_splits.csv", index=False, encoding="utf-8-sig"
    )
    all_metrics, timing_rows = [], []
    for scheme_name in FWK_SCHEMES_TO_RUN:
        package = fwk_data[scheme_name]
        scheme_dir = FWK_RUN_DIR / scheme_name
        scheme_dir.mkdir(parents=True, exist_ok=True)
        train = package["splits"]["Train"]
        training_started = time.perf_counter()
        if RUN_CV:
            best_params, cv_summary, cv_folds_detail = _tune_mixed(package)
            _boundary_warning(best_params, SIG2_GRID, GAMMA_GRID, "mixed")
            params_mode = "mixedCV_joint"
        else:
            best_params = dict(GLOBAL_PARAMS)
            params_mode = "global_fixed_from_original_notebook"
        joint_model = new_fwk(package, **best_params)
        joint_model.fit(train["X"], train["Y"], train["story_raw"])
        training_seconds = time.perf_counter() - training_started
        tables, metric_rows, pred_times = ({}, [], {})
        for split_name, arrays in package["splits"].items():
            started = time.perf_counter()
            pred_scaled = joint_model.predict(
                arrays["X"], arrays["story_raw"], PREDICT_BATCH_SIZE
            )
            prediction_seconds = time.perf_counter() - started
            pred_times[split_name] = prediction_seconds
            pred_raw = package["scaler_y"].inverse_transform(pred_scaled)
            table = pd.DataFrame({"sample_id": arrays["sample_id"]})
            for j, target in enumerate(target_cols):
                true, pred = (arrays["y_raw"][:, j], pred_raw[:, j])
                table[f"{target}_true"], table[f"{target}_pred"] = (true, pred)
                metric_rows.append(
                    {
                        "model": FWK_MODEL_NAME,
                        "scheme": scheme_name,
                        "dataset": split_name,
                        "target": target,
                        "R2": r2_score(true, pred),
                        "RMSE": np.sqrt(mean_squared_error(true, pred)),
                        "MAE": mean_absolute_error(true, pred),
                        "params": json.dumps(best_params, ensure_ascii=False),
                        "params_mode": params_mode,
                        "M": M,
                        "shared_basis": True,
                        "n_bases_total": M,
                        "prototype_selection": "kmeans++",
                    }
                )
            tables[split_name] = table.sort_values("sample_id").reset_index(drop=True)
        timing_record = {
            "Train_Time_s": training_seconds,
            "Pred_Time_Test1_s": pred_times["Test1"],
            "Pred_Time_Test2_s": pred_times["Test2"],
        }
        timing_rows.append({"Scheme": scheme_name, **timing_record})
        if RUN_CV:
            with pd.ExcelWriter(scheme_dir / "cv_results_joint.xlsx") as writer:
                cv_summary.to_excel(writer, sheet_name="CV_Summary", index=False)
                cv_folds_detail.to_excel(writer, sheet_name="CV_Folds", index=False)
        metrics = pd.DataFrame(metric_rows)
        with pd.ExcelWriter(scheme_dir / "results.xlsx") as writer:
            metrics.to_excel(writer, sheet_name="Metrics", index=False)
            pd.DataFrame([timing_record]).to_excel(
                writer, sheet_name="Timing", index=False
            )
            for split_name, table in tables.items():
                table.to_excel(writer, sheet_name=split_name, index=False)
        filenames = {
            "Train": "train_predictions",
            "Test1": "test1_internal_predictions",
            "Test2": "test2_ood_predictions",
        }
        for split_name, table in tables.items():
            table.to_excel(
                scheme_dir / f"{filenames[split_name]}({FWK_MODEL_NAME}).xlsx",
                index=False,
            )
        var_report = joint_model.variable_weight_report()
        with pd.ExcelWriter(scheme_dir / "feature_weights.xlsx") as writer:
            pd.DataFrame(var_report, columns=["input_variable", "weight"]).to_excel(
                writer, sheet_name="Variable_Weights", index=False
            )
            kernel_coord = pd.DataFrame(
                {
                    "kernel_coordinate": package["num_cols"][
                        : len(package["num_cols"]) - len(package["im_cols"])
                    ]
                    + list(joint_model.derived_names_)
                    + package["num_cols"][
                        len(package["num_cols"]) - len(package["im_cols"]) :
                    ]
                    + onehot_names,
                    "weight": joint_model.feature_weights,
                }
            )
            kernel_coord.to_excel(
                writer, sheet_name="Kernel_Coordinates_internal", index=False
            )
        state = {
            "Xs": joint_model.Xs,
            "beta": joint_model.beta,
            "b": joint_model.b,
            "feature_weights": joint_model.feature_weights,
            "sig2": joint_model.sig2,
            "gamma": joint_model.gamma,
            "shared_basis_across_outputs": True,
            "joint_solve": True,
            "prototype_selection": "kmeans++",
            "n_features_in_": joint_model.n_features_in_,
            "n_features_aug_": joint_model.n_features_aug_,
            "derived_names": list(joint_model.derived_names_),
            "derived_parents": joint_model.derived_parents_,
            "d_mean": joint_model.d_mean_,
            "d_std": joint_model.d_std_,
            "scaler_x_mean": package["scaler_x"].mean_,
            "scaler_x_std": package["scaler_x"].scale_,
            "num_names": package["num_cols"],
            "feature_names": package["feature_names"],
            "target_cols": target_cols,
        }
        joblib.dump(state, scheme_dir / "model_state_joint.joblib")
        joblib.dump(package["scaler_x"], scheme_dir / "scaler_X.pkl")
        joblib.dump(package["scaler_y"], scheme_dir / "scaler_Y.pkl")
        all_metrics.append(metrics)
        del joint_model
        gc.collect()
    fwk_summary = pd.concat(all_metrics, ignore_index=True)
    fwk_timing_summary = pd.DataFrame(timing_rows)
    with pd.ExcelWriter(FWK_RUN_DIR / "all_schemes_summary.xlsx") as writer:
        fwk_summary.to_excel(writer, sheet_name="Metrics", index=False)
        fwk_timing_summary.to_excel(writer, sheet_name="Timing", index=False)

## 5. Results in Jupyter

In [ ]:
metric_columns = ["dataset", "target", "R2", "RMSE", "MAE"]
display(
    fwk_summary.loc[
        fwk_summary["dataset"].isin(["Test1", "Test2"]), metric_columns
    ].reset_index(drop=True),
    fwk_timing_summary[["Train_Time_s", "Pred_Time_Test1_s", "Pred_Time_Test2_s"]],
)
prediction_tables = tables

## 6. Reference versus predicted response

Read saved prediction tables and render Test1/Test2 comparisons. Fixed manuscript axis limits are retained; they can clip observations beyond the plotted range.
Both figures are rendered in the notebook and also exported as PNG and PDF.


In [ ]:
def hide_zero_formatter(x, pos, integer=False):
    if np.isclose(x, 0):
        return ""
    return f"{int(x)}" if integer else f"{x:.2f}"


def nice_step(raw):
    """Round a tick interval to a multiple of 1, 2, 5 or 10."""
    exp = np.floor(np.log10(raw))
    frac = raw / 10**exp
    nice = 1 if frac < 1.5 else (2 if frac < 3.5 else (5 if frac < 7.5 else 10))
    return nice * 10**exp


def axis_settings(label, values):
    """Return axis limits and ticks, using fixed settings when available."""
    cfg = AXIS_CONFIG.get(label)
    if cfg is not None:
        return cfg["limit"], cfg["step"], cfg["integer"], cfg["diag_end"]
    vmax = float(np.max(values)) * 1.05
    step = nice_step(vmax / 4)
    limit = np.ceil(vmax / step) * step
    integer = limit >= 10
    diag_end = np.ceil(vmax / step) * step + 2 * step
    return limit, step, integer, diag_end

In [ ]:
if RUN_PREDICTION_PLOTS:
    fig_dir.mkdir(parents=True, exist_ok=True)

    TEST1_FILE = f"test1_internal_predictions({MODEL_NAME}).xlsx"

    TEST2_FILE = f"test2_ood_predictions({MODEL_NAME}).xlsx"

    scheme_dirs = sorted(
        d
        for d in data_root.iterdir()
        if d.name in FWK_SCHEMES_TO_RUN
        and d.is_dir()
        and (d / TEST1_FILE).exists()
        and (d / TEST2_FILE).exists()
    )

    if not scheme_dirs:
        raise FileNotFoundError(
            f"No scheme directories in {data_root} containing {TEST1_FILE}; run training first."
        )

    schemes = [d.name for d in scheme_dirs]

    for scheme_dir, scheme_name in zip(scheme_dirs, schemes):
        df1 = pd.read_excel(scheme_dir / TEST1_FILE)
        df2 = pd.read_excel(scheme_dir / TEST2_FILE)

        for label in TARGET_LABELS:
            y1_true = df1[f"{label}_true"].to_numpy()
            y1_pred = df1[f"{label}_pred"].to_numpy()
            y2_true = df2[f"{label}_true"].to_numpy()
            y2_pred = df2[f"{label}_pred"].to_numpy()

            r2_test1 = r2_score(y1_true, y1_pred)
            r2_test2 = r2_score(y2_true, y2_pred)

            limit, step, integer, diag_end = axis_settings(
                label, np.concatenate([y1_true, y1_pred, y2_true, y2_pred])
            )
            formatter = FuncFormatter(
                lambda x, pos, i=integer: hide_zero_formatter(x, pos, i)
            )

            fig, ax = plt.subplots(figsize=(7, 6))

            ax.scatter(
                y1_true,
                y1_pred,
                c="green",
                s=40,
                marker="o",
                alpha=0.6,
                edgecolors="k",
                linewidths=0.5,
                label=rf"Testing set1: $R^2$ = {r2_test1:.3f}",
            )
            ax.scatter(
                y2_true,
                y2_pred,
                c="red",
                s=40,
                marker="s",
                alpha=0.6,
                edgecolors="k",
                linewidths=0.5,
                label=rf"Testing set2: $R^2$ = {r2_test2:.3f}",
            )

            ax.plot([0, diag_end], [0, diag_end], color="black", linewidth=1.5)
            ax.annotate(
                "0",
                xy=(0, 0),
                xycoords="axes fraction",
                xytext=(-7, -7),
                textcoords="offset points",
                ha="right",
                va="top",
                fontsize=20,
                annotation_clip=False,
            )

            ax.set_xlim(0, limit)
            ax.set_ylim(0, limit)
            ax.xaxis.set_major_locator(MultipleLocator(step))
            ax.yaxis.set_major_locator(MultipleLocator(step))
            ax.xaxis.set_major_formatter(formatter)
            ax.yaxis.set_major_formatter(formatter)

            ax.tick_params(
                axis="both",
                which="major",
                labelsize=20,
                direction="in",
                top=True,
                right=True,
                length=5,
            )
            for tick in ax.get_xticklabels() + ax.get_yticklabels():
                tick.set_fontname("Times New Roman")

            ax.set_xlabel(
                f"Reference Value ({label})", fontsize=22, fontname="Times New Roman"
            )
            ax.set_ylabel(
                f"Predictive Value ({label})", fontsize=22, fontname="Times New Roman"
            )
            ax.grid(True, linestyle="--", alpha=0.5)

            ax.legend(
                loc="upper left",
                prop={"family": "Times New Roman", "size": 18},
                frameon=True,
                shadow=False,
                framealpha=0.9,
                edgecolor="gray",
            )

            plt.tight_layout()
            fig_name = f"{MODEL_NAME}_Comparison_{label}_{scheme_name}"
            plt.savefig(fig_dir / f"{fig_name}.png", dpi=300)
            plt.savefig(fig_dir / f"{fig_name}.pdf", dpi=300)
            display(fig)
            plt.close(fig)

## 7. Basis-vector convergence

Five-fold cross-validation uses only the training set, with identical folds for all basis counts and fixed `GLOBAL_PARAMS`. Scaling, feature weighting and basis construction are fitted within each fold.

Lines show mean validation scores; bars show changes from the preceding basis count: $\Delta R^2 = R^2_j-R^2_{j-1}$, $\Delta\mathrm{RMSE} = \mathrm{RMSE}_{j-1}-\mathrm{RMSE}_j$, and $\Delta\mathrm{MAE} = \mathrm{MAE}_{j-1}-\mathrm{MAE}_j$. Positive changes indicate improvement. The dashed line marks the configured basis count.


In [ ]:
def summarize_basis_folds(fold_scores):
    keys = ["scheme", "S", "target"]
    summary = (
        fold_scores.groupby(keys, as_index=False)[["R2", "RMSE", "MAE"]]
        .mean()
        .sort_values(keys)
        .reset_index(drop=True)
    )
    for metric in ["R2", "RMSE", "MAE"]:
        difference = summary.groupby(["scheme", "target"])[metric].diff()
        summary[f"Delta_{metric}"] = difference if metric == "R2" else -difference
    comparisons = []
    for (scheme, target), group in summary.groupby(["scheme", "target"]):
        if not {1500, 3000}.issubset(set(group["S"])):
            continue
        initial = group.loc[group["S"] == 1500].iloc[0]
        final = group.loc[group["S"] == 3000].iloc[0]
        comparisons.append(
            {
                "scheme": scheme,
                "target": target,
                "R2_1500": initial["R2"],
                "R2_3000": final["R2"],
                "Delta_R2": final["R2"] - initial["R2"],
                "RMSE_1500": initial["RMSE"],
                "RMSE_3000": final["RMSE"],
                "RMSE_Reduction_pct": 100
                * (initial["RMSE"] - final["RMSE"])
                / initial["RMSE"],
                "MAE_1500": initial["MAE"],
                "MAE_3000": final["MAE"],
                "MAE_Reduction_pct": 100
                * (initial["MAE"] - final["MAE"])
                / initial["MAE"],
            }
        )
    return summary, pd.DataFrame(comparisons)


def run_basis_convergence(package, scheme):
    train = package["splits"]["Train"]
    folds = list(
        KFold(n_splits=BASIS_CV_FOLDS, shuffle=True, random_state=TRAIN_SEED).split(
            train["Y"]
        )
    )
    if max(BASIS_COUNTS) > min(len(fit_ids) for fit_ids, _ in folds):
        raise ValueError(
            "A basis count exceeds the number of samples in a CV fitting fold."
        )
    fold_data = []
    for fit_ids, valid_ids in folds:
        fold_scaler_x = StandardScaler().fit(train["x_num_raw"][fit_ids])
        fold_scaler_y = StandardScaler().fit(train["y_raw"][fit_ids])
        fold_package = {**package, "scaler_x": fold_scaler_x, "scaler_y": fold_scaler_y}
        x_fit = np.column_stack(
            [
                fold_scaler_x.transform(train["x_num_raw"][fit_ids]),
                train["x_onehot"][fit_ids],
            ]
        )
        x_valid = np.column_stack(
            [
                fold_scaler_x.transform(train["x_num_raw"][valid_ids]),
                train["x_onehot"][valid_ids],
            ]
        )
        fold_data.append(
            (
                fold_package,
                x_fit,
                fold_scaler_y.transform(train["y_raw"][fit_ids]),
                train["story_raw"][fit_ids],
                x_valid,
                train["story_raw"][valid_ids],
                train["y_raw"][valid_ids],
            )
        )
    records = []
    with compute_runtime():
        for basis_count in BASIS_COUNTS:
            for fold, (
                fold_package,
                x_fit,
                y_fit,
                story_fit,
                x_valid,
                story_valid,
                truth,
            ) in enumerate(fold_data, 1):
                model = new_fwk(fold_package, **GLOBAL_PARAMS)
                model.M = basis_count
                model.fit(x_fit, y_fit, story_fit)
                estimate = fold_package["scaler_y"].inverse_transform(
                    model.predict(x_valid, story_valid, PREDICT_BATCH_SIZE)
                )
                for index, target in enumerate(target_cols):
                    records.append(
                        {
                            "scheme": scheme,
                            "S": basis_count,
                            "fold": fold,
                            "target": target,
                            "R2": r2_score(truth[:, index], estimate[:, index]),
                            "RMSE": np.sqrt(
                                mean_squared_error(truth[:, index], estimate[:, index])
                            ),
                            "MAE": mean_absolute_error(
                                truth[:, index], estimate[:, index]
                            ),
                        }
                    )
                del model
            print(f"Basis-vector CV completed: S={basis_count}")
    return pd.DataFrame(records)


if RUN_BASIS_CONVERGENCE:
    basis_fold_scores = pd.concat(
        [
            run_basis_convergence(fwk_data[scheme], scheme)
            for scheme in FWK_SCHEMES_TO_RUN
        ],
        ignore_index=True,
    )
    basis_summary, basis_comparison = summarize_basis_folds(basis_fold_scores)
    FWK_RUN_DIR.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(FWK_RUN_DIR / "basis_vector_convergence.xlsx") as writer:
        basis_summary.to_excel(writer, sheet_name="CV_Means", index=False)
        basis_fold_scores.to_excel(writer, sheet_name="CV_Folds", index=False)
        basis_comparison.to_excel(
            writer, sheet_name="Comparison_1500_3000", index=False
        )
    display(basis_summary, basis_comparison)


In [ ]:
if RUN_BASIS_CONVERGENCE:
    basis_summary = pd.read_excel(
        FWK_RUN_DIR / "basis_vector_convergence.xlsx", sheet_name="CV_Means"
    )
    fig_dir.mkdir(parents=True, exist_ok=True)
    for scheme in FWK_SCHEMES_TO_RUN:
        for target, color, marker in [
            ("MIDR", "tab:red", "o"),
            ("MFA", "tab:blue", "s"),
        ]:
            curve = basis_summary.loc[
                (basis_summary["scheme"] == scheme)
                & (basis_summary["target"] == target)
            ].sort_values("S")
            for metric, label in [("R2", r"$R^2$"), ("RMSE", "RMSE"), ("MAE", "MAE")]:
                fig, ax = plt.subplots(figsize=(7, 5))
                gain_ax = ax.twinx()
                gains = curve.iloc[1:]
                gain_ax.bar(
                    gains["S"],
                    gains[f"Delta_{metric}"],
                    width=55,
                    color=color,
                    alpha=0.22,
                    zorder=1,
                )
                gain_ax.axhline(0, color="gray", linewidth=0.7)
                ax.plot(
                    curve["S"],
                    curve[metric],
                    color=color,
                    marker=marker,
                    markersize=5,
                    linewidth=1.5,
                )
                ax.axvline(M, color="gray", linestyle="--", linewidth=1)
                ax.set_xlabel(r"$S$", fontsize=18)
                unit = r" (m/s$^2$)" if target == "MFA" and metric != "R2" else ""
                ax.set_ylabel(f"{label} ({target}){unit}", fontsize=18)
                gain_ax.set_ylabel(rf"$\Delta$ {label}{unit}", fontsize=18)
                ax.set_xticks([100, 500, 1000, 1500, 2000, 2500, 3000])
                ax.set_xlim(0, max(BASIS_COUNTS) * 1.04)
                ax.tick_params(labelsize=12, direction="in", top=True)
                gain_ax.tick_params(labelsize=12, direction="in")
                ax.grid(True, linestyle="--", alpha=0.3)
                ax.ticklabel_format(
                    axis="y", style="sci", scilimits=(-3, 3), useMathText=True
                )
                gain_ax.yaxis.set_major_formatter(
                    FuncFormatter(lambda value, _: f"{value:g}")
                )
                fig.tight_layout()
                filename = f"{MODEL_NAME}_Basis_Convergence_{metric}_{target}_{scheme}"
                for extension in ["png", "pdf"]:
                    fig.savefig(fig_dir / f"{filename}.{extension}", dpi=300)
                display(fig)
                plt.close(fig)
